In [1]:
import torch

from constants import DATA_FILE_BREAST_CANCER, COLUMN_NAMES, FEATURE_NAMES, LABEL_NAME, SEED
import net
from utils import read_data_to_df, create_sparsity
from net import (
    DNN, BreastCancerDataset,
    train, test
)
import torch.optim as optim
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

In [12]:
df = read_data_to_df(DATA_FILE_BREAST_CANCER, COLUMN_NAMES)

X = df[FEATURE_NAMES].values
y = df[LABEL_NAME].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

#describe the data
print("Training set:")
print(f"  Features shape: {X_train.shape}")
print(f"  Labels shape: {y_train.shape}")
print("Test set:")
print(f"  Features shape: {X_test.shape}")
print(f"  Labels shape: {y_test.shape}")
# Check for class imbalance
unique, counts = np.unique(y_train, return_counts=True)
print("Class distribution in training set:")
for cls, count in zip(unique, counts):
    print(f"  Class {cls}: {count} samples ({count / len(y_train) * 100:.2f}%)")

print("Class distribution in test set:")
unique, counts = np.unique(y_test, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  Class {cls}: {count} samples ({count / len(y_test) * 100:.2f}%)")


Training set:
  Features shape: (455, 30)
  Labels shape: (455,)
Test set:
  Features shape: (114, 30)
  Labels shape: (114,)
Class distribution in training set:
  Class B: 286 samples (62.86%)
  Class M: 169 samples (37.14%)
Class distribution in test set:
  Class B: 71 samples (62.28%)
  Class M: 43 samples (37.72%)


In [3]:
# test the tree embedding function and gbr model on train test split
from utils import read_data_to_df, create_sparsity
from tree_embeddings import create_tree_embeddings

# test imports
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb

df = read_data_to_df(DATA_FILE_BREAST_CANCER, COLUMN_NAMES)
df_sparse = create_sparsity(df, FEATURE_NAMES, sparsity_level=0.4, random_state=SEED)

df_train, df_test = train_test_split(df_sparse, test_size=0.2, random_state=SEED)

gbr, embeddings = create_tree_embeddings(df_train)

# Use DMatrix for predictions with XGBoost
dtest = xgb.DMatrix(df_test[FEATURE_NAMES].values)
gbr_predictions = gbr.predict(dtest)

# Encode labels: B (benign) = 0, M (malignant) = 1
y_test_encoded = np.where(df_test[LABEL_NAME].values == 'M', 1, 0)

gbr_auc = roc_auc_score(y_test_encoded, gbr_predictions)
gbr_ap = average_precision_score(y_test_encoded, gbr_predictions)
print(f"Gradient Boosting Classifier - AUC: {gbr_auc:.4f}, AP: {gbr_ap:.4f}")


Gradient Boosting Classifier - AUC: 0.9669, AP: 0.9684


In [ ]:
# test a single CART tree
from sklearn.tree import DecisionTreeRegressor
cart = DecisionTreeRegressor(random_state=SEED)
cart.fit(X_train, y_train)
